# Similarity Is All You Need: prototype

A training-free "find similar" engine. Pretrained models turn images and text into vectors,
and search becomes "find the nearest vectors". No model is trained.

- **Part 1:** image search on a subset of Flowers-102 with CLIP
- **Part 2:** text search on a subset of 20 Newsgroups with Sentence-BERT
- **Part 3:** swap the model with a one-line change

Run it in Google Colab: **Runtime > Run all**. A GPU is optional
(Runtime > Change runtime type > T4 GPU makes encoding faster).

In [ ]:
!pip -q install sentence-transformers

import numpy as np
import matplotlib.pyplot as plt
from typing import Protocol
from PIL import Image
from sentence_transformers import SentenceTransformer

rng = np.random.default_rng(0)

## Object-oriented part: one `Encoder` interface
Every model hides behind the same `encode()` method. The search and evaluation code
only talks to this interface, so it never knows which model is underneath.

In [ ]:
class Encoder(Protocol):
    def encode(self, items) -> np.ndarray: ...


class ClipEncoder(Encoder):
    """CLIP: images and text land in the same vector space."""

    def __init__(self, model_name: str = "clip-ViT-B-32"):
        self.model = SentenceTransformer(model_name)

    def encode(self, items) -> np.ndarray:
        items = list(items)
        return self.model.encode(items, batch_size=32, convert_to_numpy=True,
                                 show_progress_bar=len(items) > 32)


class SbertEncoder(Encoder):
    """Sentence-BERT: strong text embeddings."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def encode(self, items) -> np.ndarray:
        items = list(items)
        return self.model.encode(items, batch_size=32, convert_to_numpy=True,
                                 show_progress_bar=len(items) > 32)

## Functional part: pure, composable steps
Each function takes arrays in and returns new arrays. No hidden state, no side effects.

In [ ]:
def normalize(v):
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / n


def top_k(index, q, k=10):
    return np.argsort(-(index @ q.ravel()))[:k]


def search(encoder, index, query, k=5):
    return top_k(index, normalize(encoder.encode([query])), k)


def relevance_matrix(index, labels):
    """Row i: is each ranked result (query i excluded) the same class as query i?"""
    sims = index @ index.T
    np.fill_diagonal(sims, -np.inf)
    order = np.argsort(-sims, axis=1)[:, :-1]
    return labels[order] == labels[:, None]


def precision_at_k(rel, k):
    return rel[:, :k].mean()


def recall_at_k(rel, k):
    return (rel[:, :k].sum(axis=1) / np.maximum(rel.sum(axis=1), 1)).mean()


def mean_average_precision(rel):
    hits = np.cumsum(rel, axis=1)
    ranks = np.arange(1, rel.shape[1] + 1)
    ap = (hits / ranks * rel).sum(axis=1) / np.maximum(rel.sum(axis=1), 1)
    return ap.mean()


def report(rel, ks=(1, 5, 10)):
    for k in ks:
        print(f"Precision@{k}: {precision_at_k(rel, k):.3f}   Recall@{k}: {recall_at_k(rel, k):.3f}")
    print(f"mAP: {mean_average_precision(rel):.3f}")

# Part 1: image search (Flowers-102 + CLIP)
Prototype subset: 10 flower species, up to 30 images each.

In [ ]:
from torchvision.datasets import Flowers102

FLOWER_NAMES = ["pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea",
                "english marigold", "tiger lily", "moon orchid", "bird of paradise",
                "monkshood", "globe thistle"]
PER_CLASS = 30

flowers = Flowers102(root="data", split="test", download=True)
all_labels = np.array(flowers._labels)

picked = []
for c in range(len(FLOWER_NAMES)):
    idx = np.flatnonzero(all_labels == c)
    picked.extend(rng.choice(idx, size=min(PER_CLASS, len(idx)), replace=False))

images = [Image.open(flowers._image_files[i]).convert("RGB") for i in picked]
image_labels = all_labels[picked]
print(f"{len(images)} images, {len(FLOWER_NAMES)} classes")

In [ ]:
clip = ClipEncoder()
image_index = normalize(clip.encode(images))  # offline step: encode the dataset once
print("Index shape:", image_index.shape)

In [ ]:
def show_results(hits, query_image=None, query_text=None, query_label=None):
    panels = ([query_image] if query_image is not None else []) + [images[i] for i in hits]
    fig, axes = plt.subplots(1, len(panels), figsize=(3 * len(panels), 3.6))
    for ax, img in zip(axes, panels):
        ax.imshow(img)
        ax.axis("off")
    start = 0
    if query_image is not None:
        axes[0].set_title(f"QUERY\n{FLOWER_NAMES[query_label]}", fontweight="bold")
        start = 1
    for ax, i in zip(axes[start:], hits):
        lab = image_labels[i]
        verdict = "" if query_label is None else ("  [match]" if lab == query_label else "  [miss]")
        ax.set_title(f"{FLOWER_NAMES[lab]}{verdict}", fontsize=10)
    if query_text:
        fig.suptitle(f'Text query: "{query_text}"', fontweight="bold")
    plt.tight_layout()
    plt.show()


# Image -> image: pick any query image, get the 5 most similar ones
QUERY = 45
hits = [i for i in search(clip, image_index, images[QUERY], k=6) if i != QUERY][:5]
show_results(hits, query_image=images[QUERY], query_label=image_labels[QUERY])

In [ ]:
# Text -> image: CLIP puts text and images in the same space, so the same index works
QUERY_TEXT = "a photo of an orange tiger lily"
show_results(search(clip, image_index, QUERY_TEXT, k=5), query_text=QUERY_TEXT)

In [ ]:
# Evaluation: every image is used as a query once; a result is correct if it is the same species
print("IMAGE SEARCH (CLIP)")
report(relevance_matrix(image_index, image_labels))

# Part 2: text search (20 Newsgroups + Sentence-BERT)
Prototype subset: 5 topics, 60 posts each.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

TOPICS = ["comp.graphics", "rec.sport.baseball", "sci.med", "sci.space", "talk.politics.guns"]
PER_TOPIC = 60

news = fetch_20newsgroups(subset="test", categories=TOPICS,
                          remove=("headers", "footers", "quotes"))
news_labels = np.array(news.target)

picked = []
for c in range(len(news.target_names)):
    idx = [i for i in np.flatnonzero(news_labels == c) if len(news.data[i].strip()) > 200]
    picked.extend(rng.choice(idx, size=min(PER_TOPIC, len(idx)), replace=False))

texts = [news.data[i].strip() for i in picked]
text_labels = news_labels[picked]
print(f"{len(texts)} posts, {len(news.target_names)} topics")

In [ ]:
sbert = SbertEncoder()
text_index = normalize(sbert.encode(texts))
print("Index shape:", text_index.shape)

In [ ]:
QUERY_TEXT = "NASA is planning a new mission to send astronauts to Mars"
print(f'Query: "{QUERY_TEXT}"\n')
for rank, i in enumerate(search(sbert, text_index, QUERY_TEXT, k=3), start=1):
    snippet = " ".join(texts[i].split())[:160]
    print(f"{rank}. [{news.target_names[text_labels[i]]}] {snippet}...\n")

In [ ]:
print("TEXT SEARCH (Sentence-BERT)")
report(relevance_matrix(text_index, text_labels))

# Part 3: swap the model in one line
Same search and evaluation code, different `Encoder`. This is what makes the engine
modality-agnostic. (CLIP reads short text only, so posts are cut to 200 characters.)

In [ ]:
clip_text_index = normalize(clip.encode([t[:200] for t in texts]))

print("TEXT SEARCH with CLIP instead of Sentence-BERT")
report(relevance_matrix(clip_text_index, text_labels))